In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

import keras
import keras.backend as K
from keras.layers import Dense, GlobalAveragePooling1D, Embedding, Dropout
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.preprocessing.sequence import pad_sequences
from keras.preprocessing.text import Tokenizer
from keras.utils import to_categorical

from sklearn.model_selection import train_test_split
np.random.seed(7)


2026-01-07 06:09:40.413036: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767766180.427302      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767766180.431583      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 06:09:40.449805: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

ModuleNotFoundError: No module named 'keras.preprocessing.text'

In [2]:
df = pd.read_csv('./../input/train.csv')
df_test = pd.read_csv('./../input/test.csv')
df_full = df.append(df_test, sort=False)

a2c = {'EAP': 0, 'HPL' : 1, 'MWS' : 2}
y = np.array([a2c[a] for a in df.author])
y = to_categorical(y)
y


AttributeError: 'DataFrame' object has no attribute 'append'

In [3]:
counter = {name : defaultdict(int) for name in set(df.author)}
for (text, author) in zip(df.text, df.author):
    text = text.replace(' ', '')
    for c in text:
        counter[author][c] += 1

chars = set()
for v in counter.values():
    chars |= v.keys()
    
names = [author for author in counter.keys()]

print('c ', end='')
for n in names:
    print(n, end='   ')
print()
for c in chars:    
    print(c, end=' ')
    for n in names:
        print(counter[n][c], end=' ')
    print()


c MWS   HPL   EAP   
Z 2 43 15 
l 25074 27162 31655 
K 30 156 75 
H 606 657 791 
T 1121 1423 1986 
é 0 15 43 
S 521 757 651 
t 57079 55884 73704 
g 11366 13426 14459 
z 359 475 566 
ô 0 0 7 
W 607 649 671 
, 10863 7717 15651 
j 609 378 614 
" 1337 461 2678 
ë 0 10 0 
a 49813 50984 61285 
y 13403 11272 15188 
F 219 236 337 
: 299 44 163 
æ 0 10 32 
Ο 0 1 0 
E 408 247 391 
R 348 209 233 
ï 0 7 0 
v 7165 5843 8571 
Æ 0 4 1 
V 51 61 140 
u 18966 17463 23540 
ä 0 3 1 
A 847 1041 1120 
e 88221 79256 102699 
n 45353 45649 56062 
w 14485 14050 15625 
ü 0 5 1 
k 3338 4746 3814 
G 226 292 280 
b 8664 9585 11851 
' 426 1573 1176 
â 0 0 6 
Σ 0 1 0 
M 370 580 971 
J 60 193 145 
X 3 5 14 
q 610 681 918 
? 368 153 457 
U 43 80 151 
à 0 0 8 
Ν 0 1 0 
d 31906 29982 32981 
x 1139 930 1766 
P 314 287 382 
Y 212 94 247 
B 355 470 755 
î 0 0 1 
N 182 316 368 
Π 0 1 0 
ê 0 2 26 
; 2393 1028 1211 
c 16196 16502 21605 
ö 0 2 12 
m 18513 15879 20378 
C 275 390 372 
D 208 300 443 
s 41589 39466 48003 
I 4464 31

In [4]:
def preprocess(text):
    text = text.replace("' ", " ' ")
    signs = set(',.:;"?!')
    prods = set(text) & signs
    if not prods:
        return text

    for sign in prods:
        text = text.replace(sign, ' {} '.format(sign) )
    return text


In [5]:
def create_docs(df, n_gram_max=2):
    def add_ngram(q, n_gram_max):
            ngrams = []
            for n in range(2, n_gram_max+1):
                for w_index in range(len(q)-n+1):
                    ngrams.append('--'.join(q[w_index:w_index+n]))
            return q + ngrams
        
    docs = []
    for doc in df.text:
        doc = preprocess(doc).split()
        docs.append(' '.join(add_ngram(doc, n_gram_max)))
    
    return docs


In [6]:
min_count = 1

docs = create_docs(df)
docs_full = create_docs(df_full)
tokenizer = Tokenizer(lower=True, filters='')
tokenizer.fit_on_texts(docs_full)
num_words = sum([1 for _, v in tokenizer.word_counts.items() if v >= min_count])
num_words


NameError: name 'df_full' is not defined

In [7]:
tokenizer = Tokenizer(num_words=num_words, lower=True,  filters='')
tokenizer.fit_on_texts(docs_full)
docs = tokenizer.texts_to_sequences(docs)
print("Samples Number:", len(docs))
print("Sample 1:\n{}\nSample 1:\n{}".format(docs[0], docs[1]))


NameError: name 'Tokenizer' is not defined

In [8]:
max_size = 0
for text in docs:
    max_size = len(text) if len(text) > max_size else max_size
print("Max number of words in a sample for full dataset:", max_size)


Max number of words in a sample for full dataset: 14897


In [9]:
maxlen = max_size
docs = pad_sequences(sequences=docs, maxlen=1749)
print(len(docs))


ValueError: invalid literal for int() with base 10: 'So I did not abandon the search until I had become fully satisfied that the thief is a more astute man than myself . So--I I--did did--not not--abandon abandon--the the--search search--until until--I

In [10]:
docs_origin = create_docs(df)
type(docs_origin)
docs_origin[0]


'So I did not abandon the search until I had become fully satisfied that the thief is a more astute man than myself . So--I I--did did--not not--abandon abandon--the the--search search--until until--I I--had had--become become--fully fully--satisfied satisfied--that that--the the--thief thief--is is--a a--more more--astute astute--man man--than than--myself myself--.'

In [11]:
num_words
docs.shape


NameError: name 'num_words' is not defined

In [12]:
input_dim = np.max(docs) + 1
embedding_dims = 50
input_dim


UFuncTypeError: ufunc 'maximum' did not contain a loop with signature matching types (dtype('<U14897'), dtype('<U14897')) -> None

In [13]:
x_train, x_test, y_train, y_test = train_test_split(docs, y, test_size=0.20, random_state=42)


NameError: name 'train_test_split' is not defined

In [14]:
def create_model(embedding_dims=20, optimizer='adam'):
    
    model = Sequential()
    model.add(Embedding(input_dim=input_dim, output_dim=embedding_dims))
    model.add(GlobalAveragePooling1D())
    #model.add(Dense(15, activation='relu')) #tanh 
    #model.add(Dropout(0.5)) #relu 
    #model.add(Dense(15, activation='relu')) #tanh 
    #model.add(Dropout(0.5)) #relu 
    model.add(Dense(3, activation='softmax'))
    
    model.compile(loss='categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
    return model


In [15]:
model = create_model()
model.summary()


NameError: name 'input_dim' is not defined

In [16]:
epochs = 500
hist = model.fit(x_train, y_train,
                 batch_size=100,
                 validation_data=(x_test, y_test),
                 epochs=epochs,
                 callbacks=[EarlyStopping(patience=25, monitor='val_loss')])


NameError: name 'model' is not defined

In [17]:
"""
docs = create_docs(df)
tokenizer = Tokenizer(lower=True, filters='')
tokenizer.fit_on_texts(docs)
num_words = sum([1 for _, v in tokenizer.word_counts.items() if v >= min_count])

tokenizer = Tokenizer(num_words=num_words, lower=True, filters='')
tokenizer.fit_on_texts(docs)
docs = tokenizer.texts_to_sequences(docs)

maxlen = 256

docs = pad_sequences(sequences=docs, maxlen=maxlen)

input_dim = np.max(docs) + 1
"""


"\ndocs = create_docs(df)\ntokenizer = Tokenizer(lower=True, filters='')\ntokenizer.fit_on_texts(docs)\nnum_words = sum([1 for _, v in tokenizer.word_counts.items() if v >= min_count])\n\ntokenizer = Tokenizer(num_words=num_words, lower=True, filters='')\ntokenizer.fit_on_texts(docs)\ndocs = tokenizer.texts_to_sequences(docs)\n\nmaxlen = 256\n\ndocs = pad_sequences(sequences=docs, maxlen=maxlen)\n\ninput_dim = np.max(docs) + 1\n"

In [18]:
#model = create_model()
#model.summary()


In [19]:
test_df = pd.read_csv('../input/test.csv')
docs = create_docs(test_df)
docs = tokenizer.texts_to_sequences(docs)
docs = pad_sequences(sequences=docs, maxlen=maxlen)
y = model.predict_proba(docs)

result = pd.read_csv('../input/sample_submission.csv')
for a, i in a2c.items():
    result[a] = y[:, i]


NameError: name 'tokenizer' is not defined

In [20]:
result.to_csv('fastText_result_05.csv', index=False)


NameError: name 'result' is not defined